# Error analysis — inspect per-condition Llama runs + the merged experiment.jsonl

Investigates the UNI-65 per-condition runs from Snellius and the merged `experiment.jsonl` artifact.

| `CONDITION`              | File (relative to repo root)                                       | `condition_block.relations` keys                |
|--------------------------|--------------------------------------------------------------------|-------------------------------------------------|
| `"temporal"`             | `data/intermediate/llama_runs/temporal.jsonl`                      | `temporal_relations`                            |
| `"causal"`               | `data/intermediate/llama_runs/causal.jsonl`                        | `causal_relations`                              |
| `"temporal_causal_joint"`| `data/intermediate/llama_runs/temporal_causal_joint.jsonl`         | `joint_relations`                               |
| `"temporal_causal_independent"` | `data/intermediate/llama_runs/temporal_causal_independent.jsonl` (**composed**, no LLM call — UNI-65) | `temporal_relations`, `causal_relations` |

Change the `CONDITION` constant in cell 1 and Run All Cells to switch between them. All §1–§5 cells operate on `row["condition_block"]["relations"]`, so they handle every condition the same way. `relations` is `None` on parse-failure / ctx-overflow rows — cells guard for that.

**Hypotheses inspected (per condition):**
1. Llama is only linking *consecutive* events (eID gap == 1).
2. Label distribution collapses to a single label.
3. Cross-sentence relations are rare.
4. Coverage: large fraction of detected events are not referenced by any relation.

**§6** is the legacy n=200 stdout-log audit (UNI-24 pilot, historical) — kept for reference but no longer the primary failure-mode signal; parse errors now live in `condition_block.parse_error` on every row.

**§7** loads the merged `data/results/experiment.jsonl` and shows all four conditions side-by-side for the same summary.

In [ ]:
CONDITION = "temporal"   # change to: "temporal" | "causal" | "temporal_causal_joint" | "temporal_causal_independent"

import json, sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REL_PATH = ROOT / "data" / "intermediate" / "llama_runs" / f"{CONDITION}.jsonl"

sys.path.insert(0, str(ROOT))
from models.llama.inference import inline_events

print(f"CONDITION = {CONDITION}")
print(f"REL_PATH  = {REL_PATH}")
print(f"exists    = {REL_PATH.exists()}")

rows = [json.loads(line) for line in REL_PATH.read_text().splitlines() if line]
print(f"rows in file: {len(rows)}")

# Quick health summary for the loaded condition: source mix + parse-fail count.
src_counts = Counter(r["condition_block"]["source"] for r in rows)
parse_errors = [r for r in rows if r["condition_block"].get("parse_error")]
print(f"source mix : {dict(src_counts)}")
print(f"parse errs : {len(parse_errors)}")
for r in parse_errors:
    print(f"  - {r['wikidata_id']}/{r['summary_id']}: {r['condition_block']['parse_error']}")

## 1. Big-picture overview

In [ ]:
print(f'top-level keys (row 0): {list(rows[0].keys())}')
print(f'condition_block keys   : {list(rows[0]["condition_block"].keys())}')
rel0 = rows[0]["condition_block"]["relations"] or {}
print(f'relations keys (row 0) : {list(rel0.keys())}  (relations is None if parse failed / overflow)')

def _rel_count(cb):
    rel = cb.get("relations")
    return sum(len(v) for v in rel.values()) if rel else 0

pd.DataFrame([
    {
        'wikidata_id':    r['wikidata_id'],
        'summary_id':     r['summary_id'],
        'n_sentences':    r.get('n_sentences'),
        'n_tokens':       r.get('n_tokens'),
        'n_events':       len(r['events']),
        'source':         r['condition_block']['source'],
        'input_tokens':   r['condition_block'].get('input_tokens'),
        'output_tokens':  r['condition_block'].get('output_tokens'),
        'hit_ctx_cap':    r['condition_block'].get('hit_ctx_cap'),
        'parse_error':    r['condition_block'].get('parse_error'),
        'n_relations':    _rel_count(r['condition_block']),
    }
    for r in rows
])

## 2. Inline view — annotated summary + relations JSON

Shows each summary as Llama actually saw it: event markers `[eID|trigger|TYPE]` spliced into the text at event spans (the same `inline_events()` helper the pipeline uses). Below each annotated summary, the raw `relations` JSON Llama produced. This is the most natural way to read the output — every `source`/`target` eID in the JSON is visible as a marker inline above.

In [ ]:
from collections import defaultdict
from IPython.display import display, Markdown

for i, row in enumerate(rows):
    cb = row["condition_block"]
    rel = cb.get("relations") or {}
    n_rel = sum(len(v) for v in rel.values())

    by_sent = defaultdict(list)
    for ev in row['events']:
        by_sent[ev['sent_id']].append(ev)

    header = (
        f"### row {i} — `{row.get('summary_id','?')}` "
        f"(events={len(row['events'])}, relations={n_rel}, source=`{cb['source']}`"
        + (f", parse_error: `{cb['parse_error']}`" if cb.get('parse_error') else "")
        + ")"
    )
    md = [
        header,
        "",
        "**Annotated summary** (events inlined as `[eID|trigger|TYPE]`, bolded):",
        "",
    ]

    for sent_id, sent in enumerate(row['sentences']):
        evs = sorted(by_sent.get(sent_id, []), key=lambda e: e['start'])
        out, cursor = [], 0
        for ev in evs:
            out.append(sent[cursor:ev['start']])
            out.append(f"**[{ev['event_id']}|{ev['trigger']}|{ev['event_type']}]**")
            cursor = ev['end']
        out.append(sent[cursor:])
        md.append(f"- s{sent_id}: {''.join(out)}")

    md += [
        "",
        "**Relations (raw JSON from `condition_block.relations`):**",
        "",
        "```json",
        json.dumps(rel, indent=2) if rel else "null (parse failure or ctx overflow — see source / parse_error above)",
        "```",
    ]
    # For Llama-source rows, also surface the verbatim response_raw — useful for
    # diagnosing parse failures (you can see exactly what Llama produced).
    if cb.get("response_raw"):
        md += [
            "",
            "**Raw Llama response (`condition_block.response_raw`, first 1500 chars):**",
            "",
            "```",
            cb["response_raw"][:1500] + ("\n... (truncated)" if len(cb["response_raw"]) > 1500 else ""),
            "```",
        ]
    display(Markdown("\n".join(md)))

## 3. eID-distance distribution (the hypothesis check)

For each relation, compute `|source_id - target_id|`. If Llama only links consecutive events, every diff = 1.

In [ ]:
all_diffs = []
for row in rows:
    rel = row["condition_block"].get("relations") or {}
    for rels in rel.values():
        for r in rels:
            all_diffs.append(abs(int(r['source'][1:]) - int(r['target'][1:])))

print(f'total relations across all rows: {len(all_diffs)}')
print(f'diff distribution: {sorted(Counter(all_diffs).items())}')
if all_diffs:
    consecutive = sum(1 for d in all_diffs if d == 1)
    print(f'consecutive (diff==1): {consecutive}/{len(all_diffs)} = {consecutive/len(all_diffs):.1%}')
    print(f'max diff: {max(all_diffs)}')
    print(f'mean diff: {sum(all_diffs)/len(all_diffs):.2f}')

## 4. Per-row relations with triggers resolved (arrow view)

Same data as §2 but compressed: each relation rendered as `source (trigger) --LABEL--> target (trigger)`, with the eID gap.

In [ ]:
for i, row in enumerate(rows):
    cb = row["condition_block"]
    ev = {e['event_id']: e for e in row['events']}
    rel = cb.get("relations") or {}
    print(f"=== row {i}  summary_id={row.get('summary_id','?')}  events={len(row['events'])}  "
          f"source={cb['source']}"
          + (f"  parse_error={cb['parse_error']!r}" if cb.get('parse_error') else "")
          + " ===")
    if not rel:
        print("  (no relations — parse failure or ctx overflow)")
        print()
        continue
    for rel_type, rels in rel.items():
        print(f'  [{rel_type}]  n={len(rels)}')
        for r in rels:
            s, t = ev.get(r['source']), ev.get(r['target'])
            s_repr = f"{r['source']} ({s['trigger']!r}, sent {s['sent_id']})" if s else f"{r['source']} [MISSING]"
            t_repr = f"{r['target']} ({t['trigger']!r}, sent {t['sent_id']})" if t else f"{r['target']} [MISSING]"
            gap = abs(int(r['source'][1:]) - int(r['target'][1:]))
            print(f'    {s_repr}  --{r["relation"]}-->  {t_repr}   (eID gap = {gap})')
    print()

## 5. eID coverage

Which events actually appear in any relation? Are there events Llama ignored entirely? Any hallucinated eIDs (mentioned in `relations` but not in `events`)?

In [ ]:
for i, row in enumerate(rows):
    cb = row["condition_block"]
    all_eids = {e['event_id'] for e in row['events']}
    used_eids = set()
    rel = cb.get("relations") or {}
    for rels in rel.values():
        for r in rels:
            used_eids.add(r['source'])
            used_eids.add(r['target'])
    unused = all_eids - used_eids
    invalid = used_eids - all_eids
    suffix = f" (source={cb['source']}"
    if cb.get('parse_error'):
        suffix += f", parse_error={cb['parse_error']!r}"
    suffix += ")"
    if not rel:
        print(f"row {i}: no relations recorded{suffix}")
        continue
    print(f'row {i}: {len(used_eids & all_eids)}/{len(all_eids)} events appear in relations '
          f'({len(used_eids & all_eids)/len(all_eids):.0%} coverage); '
          f'{len(unused)} unused; {len(invalid)} hallucinated{suffix}')
    if invalid:
        print(f'  ⚠ hallucinated eIDs: {sorted(invalid)}')

## 6. n=200 audit — failure-mode inspection (UNI-24 pilot review)

The four `llama-*-n200-*.out` logs in `data/intermediate/llama_logs/` are the four-condition pilot batch from [UNI-52](https://linear.app/uva-school-thesis/issue/UNI-52/). These cells parse them, join with the input rows from `tma_subset_events_full.jsonl`, and inspect the stories that triggered each failure mode — so we can iterate on the prompts in [UNI-24](https://linear.app/uva-school-thesis/issue/UNI-24/) deliberately rather than blindly.

In [7]:
# === n=200 audit findings (UNI-24 pilot inspection) ===
# Parses the four llama-*-n200-*.out logs + joins with the input rows from
# tma_subset_events_full.jsonl so we can inspect WHICH stories triggered each
# failure category. Independent of the smoke3 CONDITION selector above.
import re

LOGS_DIR    = ROOT / "data" / "intermediate" / "llama_logs"
EVENTS_PATH = ROOT / "data" / "intermediate" / "tma_subset_events_full.jsonl"
JOB_NAME_TO_COND = {
    "temporal": "temporal",
    "causal":   "causal",
    "tcindep":  "temporal_causal_independent",
    "tcjoint":  "temporal_causal_joint",
}
SUCCESS_RE = re.compile(r"^row (\d+): input_tokens=(\d+) output_tokens=(\d+)\s*$")
ERROR_RE   = re.compile(r"^row (\d+): (?!input_tokens=)(.+)$")

def classify_error(msg: str) -> str:
    if "Unterminated string" in msg or msg.startswith("Unterminated"): return "truncation"
    if msg.startswith("Expecting value"):                              return "prose_wrap"
    if msg.startswith("unknown label"):                                return "label_hallucination"
    if msg.startswith("bad eID"):                                      return "bad_eid"
    if "Expecting property name" in msg or ("Expecting" in msg and "delimiter" in msg):
        return "json_malformed"
    return "other"

# First 200 rows of the events file = exactly what the audit jobs processed.
input_rows = []
with EVENTS_PATH.open() as f:
    for i, line in enumerate(f):
        if i >= 200: break
        input_rows.append(json.loads(line))

records = []
for log_path in sorted(LOGS_DIR.glob("llama-*-n200-*.out")):
    suffix = log_path.name.split("-")[1]
    cond = JOB_NAME_TO_COND.get(suffix)
    if cond is None: continue
    per_row: dict[int, dict] = {}
    for line in log_path.read_text().splitlines():
        if (m := SUCCESS_RE.match(line)):
            per_row[int(m.group(1))] = {
                "input_tokens":  int(m.group(2)),
                "output_tokens": int(m.group(3)),
                "error":         None,
            }
        elif (m := ERROR_RE.match(line)):
            ridx = int(m.group(1))
            if ridx in per_row:
                per_row[ridx]["error"] = m.group(2).strip()
            else:
                per_row[ridx] = {"input_tokens": None, "output_tokens": None, "error": m.group(2).strip()}
    for ridx, info in sorted(per_row.items()):
        records.append({"condition": cond, "row_idx": ridx, **info})

audit_df = pd.DataFrame(records)
audit_df["category"] = audit_df["error"].apply(lambda e: classify_error(e) if isinstance(e, str) else "ok")
audit_df["n_events"] = audit_df["row_idx"].map(lambda i: len(input_rows[i]["events"]))
audit_df["n_words"]  = audit_df["row_idx"].map(lambda i: len(input_rows[i]["text"].split()))

# Per-condition × failure-category counts + total fail %.
counts = audit_df.groupby(["condition", "category"]).size().unstack(fill_value=0)
counts["total"]    = audit_df.groupby("condition").size()
counts["fail_pct"] = (100 * (counts["total"] - counts.get("ok", 0)) / counts["total"]).round(1)
print(f"\033[1;33mn=200 audit — failure breakdown per condition:\033[0m\n")
display(counts)

n=200 audit — failure breakdown per condition:



category,bad_eid,json_malformed,label_hallucination,ok,prose_wrap,truncation,total,fail_pct
condition,,,,,,,,
causal,1,6,1,177,11,4,200,11.5
temporal,0,5,9,154,4,28,200,23.0
temporal_causal_independent,0,14,48,54,30,54,200,73.0
temporal_causal_joint,0,1,11,164,20,4,200,18.0


### 6.1 Truncation (output hit the static 2,048 cap → Unterminated string)

These should mostly disappear after the dynamic `max_new_tokens` fix shipped with UNI-52. Inspecting them here mainly tells us *which stories* are output-heavy, so we know what to expect under the dynamic cap.

In [8]:
# Cases where the LLM hit the static max_new_tokens=2048 cap and the resulting JSON was
# cut mid-string. These should mostly disappear after the dynamic max_new_tokens fix in
# infer_relations.py (UNI-52 close-out). Worth checking which story sizes triggered them.
trunc = audit_df[audit_df["category"] == "truncation"].sort_values("n_events", ascending=False)
print(f"{len(trunc)} truncation failures across conditions:")
print(trunc.groupby("condition").size().to_string())
print()
display(trunc[["condition", "row_idx", "output_tokens", "n_events", "n_words", "error"]].head(15))

# One concrete example with the input story.
if len(trunc):
    ex = trunc.iloc[0]
    r  = input_rows[int(ex["row_idx"])]
    display(Markdown(
        f"### Example truncation\n"
        f"- condition: `{ex['condition']}`\n"
        f"- row_idx: `{ex['row_idx']}` (wikidata_id `{r['wikidata_id']}`, summary_id `{r['summary_id']}`)\n"
        f"- n_events: **{len(r['events'])}**, n_words: **{len(r['text'].split())}**\n"
        f"- output_tokens: **{ex['output_tokens']}** (cap)\n"
        f"- error: `{ex['error']}`\n"
    ))

90 truncation failures across conditions:
condition
causal                          4
temporal                       28
temporal_causal_independent    54
temporal_causal_joint           4



,condition,row_idx,output_tokens,n_events,n_words,error
151,causal,151,2048,181,1647,Unterminated string starting at: line 366 colu...
551,temporal_causal_joint,151,2048,181,1647,Unterminated string starting at: line 355 colu...
759,temporal,159,2048,160,314,Unterminated string starting at: line 95 colum...
0,causal,0,2048,155,1135,Unterminated string starting at: line 366 colu...
600,temporal,0,2048,155,1135,Unterminated string starting at: line 95 colum...
753,temporal,153,2048,142,927,Unterminated string starting at: line 95 colum...
765,temporal,165,2048,134,22,Unterminated string starting at: line 95 colum...
304,temporal_causal_independent,104,2048,127,820,Unterminated string starting at: line 95 colum...
704,temporal,104,2048,127,820,Unterminated string starting at: line 95 colum...
651,temporal,51,2048,124,1101,Unterminated string starting at: line 95 colum...


### Example truncation
- condition: `causal`
- row_idx: `151` (wikidata_id `102438`, summary_id `fr`)
- n_events: **181**, n_words: **1647**
- output_tokens: **2048** (cap)
- error: `Unterminated string starting at: line 366 column 17 (char 5476)`


### 6.2 Label hallucination (LLM invented a label outside the codebook)

E.g. `unknown label: AFTER` when the temporal codebook only has BEFORE / OVERLAPS / CONTAINS / IDENTITY. The biggest concentration is in `temporal_causal_independent` (48 failures) — that condition's prompt may not constrain the LLM tightly enough.

In [ ]:
# Cases where parse_and_validate said "unknown label". Some are real hallucinations
# (label is in NEITHER codebook), but some — especially in temporal_causal_independent —
# are SECTION-MISPLACEMENT: a valid causal label (e.g. CAUSE_TO_END, CAUSE) appearing
# under the "temporal_relations" array (or vice versa). parse_and_validate iterates
# temporal first, so it flags any causal label there as "unknown".
import yaml

PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"

# Per-condition codebook lookup. NOTE: the temporal_causal_independent.yaml was deleted
# in UNI-65 (that condition is now composed post-hoc, not a Llama call). We hold its
# old codebook inline here so this historical n=200 analysis still classifies the
# section_mismatch failures it recorded.
codebooks = {}
for cond in ("temporal", "causal", "temporal_causal_joint"):
    cfg = yaml.safe_load((PROMPTS_DIR / f"{cond}.yaml").read_text())
    codebooks[cond] = {
        "temporal": set(cfg.get("allowed_temporal_labels", [])),
        "causal":   set(cfg.get("allowed_causal_labels",   [])),
        "labels":   set(cfg.get("allowed_labels",          [])),
    }
# Historical reconstruction of the pre-UNI-65 tcindep codebook (frozen, for log replay only).
codebooks["temporal_causal_independent"] = {
    "temporal": codebooks["temporal"]["labels"],
    "causal":   codebooks["causal"]["labels"],
    "labels":   set(),
}

UNIVERSE = set().union(*[
    cb["temporal"] | cb["causal"] | cb["labels"] for cb in codebooks.values()
])

def classify_unknown(cond: str, label: str) -> str:
    cb = codebooks[cond]
    own = cb["temporal"] | cb["causal"] | cb["labels"]   # everything this condition recognizes
    if label not in UNIVERSE:
        return "true_hallucination"
    if label in own:
        return "section_mismatch"
    return "wrong_condition_label"

hall = audit_df[audit_df["category"] == "label_hallucination"].copy()
hall["hallucinated_label"] = hall["error"].str.extract(r"unknown label: (\S+)")
hall["bucket"] = hall.apply(lambda r: classify_unknown(r["condition"], r["hallucinated_label"]), axis=1)

print(f"{len(hall)} 'unknown label' failures across conditions:\n")
print("By condition × bucket:")
display(hall.groupby(["condition", "bucket"]).size().unstack(fill_value=0))

print("\nLabel × bucket × condition — what's actually getting flagged?")
display(hall.groupby(["condition", "bucket", "hallucinated_label"]).size().to_frame("n").reset_index())

### 6.3 Prose-wrapped JSON (`Expecting value: line 1 column 1`)

LLM prefixed its JSON with conversational text ("Here are the relations:") before the `{`. Affects every condition. Cheap recoverable failure — `parse_and_validate` could strip everything before the first `{`, or the system prompt could be tightened to forbid prose.

In [10]:
# Cases where Llama prefixed the JSON with prose ("Here are the causal relations:") so
# json.loads errors at char 0. Cheap fix: strip everything before first { in parse_and_validate.
prose = audit_df[audit_df["category"] == "prose_wrap"]
print(f"{len(prose)} prose-wrap failures across conditions:")
print(prose.groupby("condition").size().to_string())
print()
print("Output-token distribution — prose-wrap can fire on short OR long outputs:")
display(prose.groupby("condition")["output_tokens"].agg(["count", "min", "median", "max"]).astype(int))
print()
display(prose[["condition", "row_idx", "output_tokens", "n_events", "n_words"]].head(15))

65 prose-wrap failures across conditions:
condition
causal                         11
temporal                        4
temporal_causal_independent    30
temporal_causal_joint          20

Output-token distribution — prose-wrap can fire on short OR long outputs:


,count,min,median,max
condition,,,,
causal,11,664,1613,2048
temporal,4,1891,2048,2048
temporal_causal_independent,30,768,2048,2048
temporal_causal_joint,20,473,1542,2048


,condition,row_idx,output_tokens,n_events,n_words
19,causal,19,997,134,1106
38,causal,38,2048,166,1377
83,causal,83,664,173,1393
101,causal,101,2048,106,717
125,causal,125,1613,128,1058
143,causal,143,2048,335,2662
153,causal,153,1779,142,927
159,causal,159,2048,160,314
160,causal,160,1588,171,1641
162,causal,162,1532,139,605


### 6.4 Other malformed JSON / bad eIDs

Catch-all for structural JSON errors that aren't truncation/prose-wrap (missing delimiters, bad property names) and for `bad eID:` errors from `parse_and_validate` (event IDs referenced in relations that don't exist in the events list). Usually small but worth eyeballing for systematic patterns.

In [11]:
# Structural JSON errors not covered by truncation or prose-wrap — missing delimiters,
# bad property names, etc. Smallest bucket; usually one-off Llama brain farts but worth
# eyeballing to make sure no systematic issue is hiding.
malformed = audit_df[audit_df["category"].isin(["json_malformed", "bad_eid", "other"])]
print(f"{len(malformed)} miscellaneous JSON failures across conditions:")
print(malformed.groupby(["condition", "category"]).size().unstack(fill_value=0))
print()

# Set option to display full column width for errors
pd.set_option('display.max_colwidth', None)
display(malformed[["condition", "row_idx", "category", "error", "n_events", "n_words"]].head(20))
# Reset option if needed for other cells
pd.reset_option('display.max_colwidth')

27 miscellaneous JSON failures across conditions:
category                     bad_eid  json_malformed
condition                                           
causal                             1               6
temporal                           0               5
temporal_causal_independent        0              14
temporal_causal_joint              0               1



,condition,row_idx,category,error,n_events,n_words
28,causal,28,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5484)",85,752
35,causal,35,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5486)",100,746
40,causal,40,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5486)",120,770
84,causal,84,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5482)",81,595
89,causal,89,bad_eid,bad eID: e13|start|Process_start,43,348
126,causal,126,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5478)",74,573
157,causal,157,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5492)",103,688
207,temporal_causal_independent,7,json_malformed,Expecting property name enclosed in double quotes: line 95 column 39 (char 5753),96,752
222,temporal_causal_independent,22,json_malformed,Expecting property name enclosed in double quotes: line 97 column 22 (char 5846),63,483
243,temporal_causal_independent,43,json_malformed,Expecting property name enclosed in double quotes: line 95 column 39 (char 5761),101,672


## 7. Merged `experiment.jsonl` — all four conditions per summary

This is the artifact UNI-65 produces. One row per `(wikidata_id, summary_id)` with all four condition slots nested under `conditions`. Useful for spotting cross-condition disagreements on the same summary (e.g. temporal succeeded but joint failed to parse).

In [ ]:
EXPERIMENT_PATH = ROOT / "data" / "results" / "experiment.jsonl"
print(f"EXPERIMENT_PATH = {EXPERIMENT_PATH}")
print(f"exists          = {EXPERIMENT_PATH.exists()}")

exp_rows = [json.loads(line) for line in EXPERIMENT_PATH.read_text().splitlines() if line]
print(f"rows in file    : {len(exp_rows)}\n")

# Per-row summary: which condition slots are present, with their source + parse_error.
def _summarise(row):
    out = {"wikidata_id": row["wikidata_id"], "summary_id": row["summary_id"],
           "n_events": len(row["events"]), "n_tokens": row["n_tokens"],
           "embeddings": row["embeddings"], "baselines": row["baselines"]}
    for cond in ("temporal", "causal", "temporal_causal_joint", "temporal_causal_independent"):
        cb = row["conditions"].get(cond)
        if cb is None:
            out[cond] = "(missing)"
            continue
        rel = cb.get("relations") or {}
        n_rel = sum(len(v) for v in rel.values())
        tag = cb["source"]
        if cb.get("parse_error"):
            tag += "/parse_err"
        out[cond] = f"{tag} n_rel={n_rel}"
    return out

pd.DataFrame([_summarise(r) for r in exp_rows])

In [ ]:
# Cross-condition view: pick one summary, show all four condition slots side by side.
# Pick the first row by default; change `pick_idx` to inspect a different summary.

pick_idx = 0
row = exp_rows[pick_idx]
print(f"wikidata_id={row['wikidata_id']}  summary_id={row['summary_id']}  "
      f"n_events={len(row['events'])}  n_tokens={row['n_tokens']}")
print(f"text (first 200 chars): {row['text'][:200]}{'...' if len(row['text']) > 200 else ''}\n")

for cond in ("temporal", "causal", "temporal_causal_joint", "temporal_causal_independent"):
    cb = row["conditions"].get(cond)
    print(f"=== {cond} ===")
    if cb is None:
        print("  (slot missing — condition was not produced for this summary)\n")
        continue
    print(f"  source        : {cb['source']}")
    if cb["source"] == "llama":
        print(f"  input_tokens  : {cb.get('input_tokens')}")
        print(f"  output_tokens : {cb.get('output_tokens')}")
        print(f"  hit_ctx_cap   : {cb.get('hit_ctx_cap')}")
        print(f"  parse_error   : {cb.get('parse_error')!r}")
    elif cb["source"] == "composed":
        print(f"  composed_from : {cb.get('composed_from')}")
    elif cb["source"] == "skipped_ctx_overflow":
        print(f"  input_tokens  : {cb.get('input_tokens')} (>= LLAMA_CTX — never ran)")
    rel = cb.get("relations") or {}
    if not rel:
        print("  relations     : None")
    else:
        for key, triples in rel.items():
            print(f"  {key}: {len(triples)} relations")
    print()

In [ ]:
# Full DataFrame view of experiment.jsonl — every row, every condition, flattened.
# One row per (wikidata_id, summary_id, condition) so you can filter / pivot freely.

def _flatten(exp_rows):
    out = []
    for r in exp_rows:
        for cond, cb in r["conditions"].items():
            rel = cb.get("relations") or {}
            rel_counts = {f"n_{k}": len(v) for k, v in rel.items()}
            out.append({
                "wikidata_id":    r["wikidata_id"],
                "summary_id":     r["summary_id"],
                "lang":           r["lang"],
                "n_events":       len(r["events"]),
                "n_tokens":       r["n_tokens"],
                "n_sentences":    r["n_sentences"],
                "condition":      cond,
                "source":         cb["source"],
                "input_tokens":   cb.get("input_tokens"),
                "output_tokens":  cb.get("output_tokens"),
                "max_new_tokens": cb.get("max_new_tokens"),
                "hit_ctx_cap":    cb.get("hit_ctx_cap"),
                "parse_error":    cb.get("parse_error"),
                "n_relations":    sum(rel_counts.values()),
                **rel_counts,
                "prompt_len":     len(cb.get("prompt_rendered") or ""),
                "response_len":   len(cb.get("response_raw") or ""),
            })
    return out

exp_df = pd.DataFrame(_flatten(exp_rows))
print(f"shape: {exp_df.shape}")
print()
print("dtypes:")
print(exp_df.dtypes.to_string())
print()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(exp_df)